In [1]:
import os, sys, time, json, shutil, subprocess, warnings
warnings.filterwarnings("ignore")

try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
if shutil.which("java") is None:
    subprocess.run("apt-get -qq install -y openjdk-17-jdk-headless > /dev/null", shell=True, check=True)

from google.colab import drive
drive.mount("/content/drive")
PERSIST_DIR = "/content/drive/MyDrive/7006SCN_GSOD"
SCRATCH_DIR = "/content/gsod_scratch"
LOCAL_DIR   = "/content/gsod_local"

def sp(path):
    return path

RAW_DIR     = SCRATCH_DIR + "/raw_csv"
ARCHIVE_DIR = PERSIST_DIR + "/archives"
PARQUET_RAW = LOCAL_DIR + "/parquet/gsod_raw"
FEATURES    = LOCAL_DIR + "/parquet/gsod_features"
PRED_DIR    = LOCAL_DIR + "/parquet/predictions"
MODELS_DIR  = LOCAL_DIR + "/models"
OUT_DIR     = PERSIST_DIR + "/outputs"
for d in [PERSIST_DIR, SCRATCH_DIR, LOCAL_DIR, ARCHIVE_DIR, MODELS_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

def backup_to_drive(local_path):
    dest = local_path.replace(LOCAL_DIR, PERSIST_DIR)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(local_path, dest)
    print("backed up ->", dest)

def restore_from_drive(local_path):
    if not os.path.exists(local_path):
        shutil.copytree(local_path.replace(LOCAL_DIR, PERSIST_DIR), local_path)
        print("restored <-", local_path)

YEARS       = [2020, 2021, 2022, 2023]
TRAIN_YEARS = [2020, 2021, 2022]
TEST_YEAR   = 2023
SEED        = 42

from pyspark.sql import SparkSession, functions as F, Window
from pyspark import StorageLevel
TOTAL_RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
DRIVER_MEM = f"{max(2, int(TOTAL_RAM_GB * 0.7))}g"

spark = (SparkSession.builder
         .master("local[*]")
         .appName("7006SCN_GSOD_Task3")
         .config("spark.driver.memory", DRIVER_MEM)
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def n_partitions(df):
    return df.rdd.getNumPartitions()

def safe_persist(df, name, level=StorageLevel.MEMORY_AND_DISK):
    out = df.persist(level)
    out.count()
    return out

def save_csv(pdf, name):
    path = f"{OUT_DIR}/{name}"
    pdf.to_csv(path, index=False)
    print("saved ->", path)

print(f"Spark {spark.version} | master={spark.sparkContext.master} | cores={spark.sparkContext.defaultParallelism} "
      f"| driver memory={DRIVER_MEM} | RAM={TOTAL_RAM_GB:.1f} GB")
print("PERSIST_DIR =", PERSIST_DIR)

restore_from_drive(FEATURES)
restore_from_drive(f"{MODELS_DIR}/prep_pipeline")

Mounted at /content/drive
Spark 4.0.4 | master=local[*] | cores=2 | driver memory=8g | RAM=12.7 GB
PERSIST_DIR = /content/drive/MyDrive/7006SCN_GSOD
restored <- /content/gsod_local/parquet/gsod_features
restored <- /content/gsod_local/models/prep_pipeline


In [2]:
import pandas as pd, numpy as np
from pyspark.ml import PipelineModel
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
                                       RandomForestClassifier, GBTClassifier)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

TUNE_FRACTION = float(os.environ.get("GSOD_TUNE_FRACTION", 0.03))
TEST_FRACTION = float(os.environ.get("GSOD_TEST_FRACTION", 1.0))
CV_FOLDS, CV_PARALLELISM = 3, 2

features   = spark.read.parquet(sp(FEATURES))
prep_model = PipelineModel.load(sp(f"{MODELS_DIR}/prep_pipeline"))
feature_names = json.load(open(f"{OUT_DIR}/feature_names.json"))

KEEP = ["STATION", "NAME", "DATE", "YEAR", "MONTH", "LATITUDE", "LONGITUDE", "CLIMATE_ZONE",
        "HEMISPHERE", "SEASON", "IS_EXTREME", "features", "label"]

train_all = features.filter(F.col("YEAR").isin(TRAIN_YEARS))
test_all  = features.filter(F.col("YEAR") == TEST_YEAR)
if TEST_FRACTION < 1:
    test_all = test_all.sample(fraction=TEST_FRACTION, seed=SEED)

counts = {int(r["label"]): r["count"] for r in train_all.groupBy("label").count().collect()}
n_tot = sum(counts.values())
W_POS, W_NEG = n_tot / (2 * counts[1]), n_tot / (2 * counts[0])
print(f"train rows={n_tot:,} | positive share={counts[1]/n_tot:.2%} | w_pos={W_POS:.2f} w_neg={W_NEG:.2f}")

def add_weight(d):
    return d.withColumn("w", F.when(F.col("label") == 1, W_POS).otherwise(W_NEG))

tune = add_weight(prep_model.transform(train_all.sampleBy("label", {0.0: TUNE_FRACTION, 1.0: TUNE_FRACTION}, seed=SEED))
                  .select(*KEEP))
tune = safe_persist(tune, "tune")
test = safe_persist(prep_model.transform(test_all).select(*KEEP), "test")
print(f"tuning sample rows = {tune.count():,} | test rows = {test.count():,}")

train rows=11,939,933 | positive share=14.96% | w_pos=3.34 w_neg=0.59
tuning sample rows = 358,135 | test rows = 3,959,170


In [3]:
pr_eval  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderPR")
roc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC")

def evaluate(pred, name, stage):
    cm = pred.groupBy("label", "prediction").count().toPandas()
    get = lambda l, p: int(cm[(cm.label == l) & (cm.prediction == p)]["count"].sum())
    tp, fp, tn, fn = get(1, 1), get(0, 1), get(0, 0), get(1, 0)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    return {"model": name, "stage": stage,
            "accuracy": (tp + tn) / (tp + tn + fp + fn), "precision": prec, "recall": rec,
            "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0, "specificity": spec,
            "balanced_accuracy": (rec + spec) / 2,
            "auc_roc": roc_eval.evaluate(pred), "auc_pr": pr_eval.evaluate(pred),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn}

RESULTS, TUNING_LOG, BEST = [], [], {}

def param_str(model, names):
    return ", ".join(f"{n}={model.getOrDefault(n)}" for n in names)

def run_baseline(name, est):
    t0 = time.perf_counter(); m = est.fit(tune); t = time.perf_counter() - t0
    r = evaluate(m.transform(test), name, "baseline"); r["train_time_s"] = round(t, 1)
    RESULTS.append(r)
    print(f"[{name}] BASELINE trained in {t:,.1f}s -> PR-AUC={r['auc_pr']:.4f} ROC-AUC={r['auc_roc']:.4f} "
          f"recall={r['recall']:.3f} precision={r['precision']:.3f}")
    return m

def run_tuning(name, est, grid, tuned_params):
    cv = CrossValidator(estimator=est, estimatorParamMaps=grid, evaluator=pr_eval,
                        numFolds=CV_FOLDS, parallelism=CV_PARALLELISM, seed=SEED)
    t0 = time.perf_counter(); cvm = cv.fit(tune); cv_t = time.perf_counter() - t0
    for pm, score in zip(grid, cvm.avgMetrics):
        TUNING_LOG.append({"model": name, "params": ", ".join(f"{p.name}={v}" for p, v in pm.items()),
                           "cv_mean_pr_auc": round(score, 5)})
    best_pm = grid[int(np.argmax(cvm.avgMetrics))]

    t0 = time.perf_counter(); best = est.copy(best_pm).fit(tune); fit_t = time.perf_counter() - t0
    r = evaluate(best.transform(test), name, "tuned")
    r.update({"train_time_s": round(fit_t, 1), "cv_time_s": round(cv_t, 1),
              "cv_best_pr_auc": max(cvm.avgMetrics), "hyperparameters": param_str(best, tuned_params),
              "grid_size": len(grid)})
    RESULTS.append(r); BEST[name] = best
    print(f"[{name}] CrossValidator: {len(grid)} configs x {CV_FOLDS} folds in {cv_t:,.1f}s")
    print(f"[{name}] best = {r['hyperparameters']} | trained in {fit_t:,.1f}s")
    print(f"[{name}] TEST -> PR-AUC={r['auc_pr']:.4f} ROC-AUC={r['auc_roc']:.4f} F1={r['f1']:.3f} "
          f"recall={r['recall']:.3f} precision={r['precision']:.3f}")
    return best

def save_model(name, model):
    model.write().overwrite().save(sp(f"{MODELS_DIR}/{name}"))
    (model.transform(test)
          .withColumn("prob", vector_to_array("probability")[1])
          .select("STATION", "NAME", "DATE", "MONTH", "LATITUDE", "LONGITUDE", "CLIMATE_ZONE", "HEMISPHERE",
                  "SEASON", "IS_EXTREME", "label", "prediction", "prob")
          .write.mode("overwrite").parquet(sp(f"{PRED_DIR}/{name}")))
    backup_to_drive(f"{MODELS_DIR}/{name}")
    backup_to_drive(f"{PRED_DIR}/{name}")
    print(f"saved model + test predictions for {name}")

In [4]:
pers = test.withColumn("prediction", F.col("IS_EXTREME").cast("double"))
cm = pers.groupBy("label", "prediction").count().toPandas()
get = lambda l, p: int(cm[(cm.label == l) & (cm.prediction == p)]["count"].sum())
tp, fp, tn, fn = get(1, 1), get(0, 1), get(0, 0), get(1, 0)
p_, r_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
RESULTS.append({"model": "Persistence (naive)", "stage": "reference", "accuracy": (tp + tn) / (tp + tn + fp + fn),
                "precision": p_, "recall": r_, "f1": 2 * p_ * r_ / max(p_ + r_, 1e-9),
                "specificity": tn / max(tn + fp, 1), "balanced_accuracy": (r_ + tn / max(tn + fp, 1)) / 2,
                "auc_roc": np.nan, "auc_pr": np.nan, "tp": tp, "fp": fp, "tn": tn, "fn": fn, "train_time_s": 0})
print(pd.DataFrame(RESULTS).round(4).to_string(index=False))

              model     stage  accuracy  precision  recall     f1  specificity  balanced_accuracy  auc_roc  auc_pr     tp     fp      tn     fn  train_time_s
Persistence (naive) reference    0.8899     0.6467  0.6466 0.6467       0.9348             0.7907      NaN     NaN 399062 218036 3124008 218064             0


In [5]:
lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="w", maxIter=50)
lr_base = run_baseline("LogisticRegression", lr)

[LogisticRegression] BASELINE trained in 27.2s -> PR-AUC=0.6841 ROC-AUC=0.8775 recall=0.734 precision=0.551


In [6]:
lr_grid = (ParamGridBuilder()
           .addGrid(lr.regParam, [0.001, 0.01, 0.1])
           .addGrid(lr.elasticNetParam, [0.0, 0.5])
           .build())
lr_best = run_tuning("LogisticRegression", lr, lr_grid, ["regParam", "elasticNetParam", "maxIter"])

[LogisticRegression] CrossValidator: 6 configs x 3 folds in 255.8s
[LogisticRegression] best = regParam=0.001, elasticNetParam=0.0, maxIter=50 | trained in 16.3s
[LogisticRegression] TEST -> PR-AUC=0.6837 ROC-AUC=0.8772 F1=0.630 recall=0.733 precision=0.552


In [7]:
save_model("LogisticRegression", lr_best)
coef = pd.DataFrame({"feature": feature_names, "coefficient": lr_best.coefficients.toArray()})
coef["abs"] = coef.coefficient.abs()
print(coef.sort_values("abs", ascending=False).head(12).drop(columns="abs").to_string(index=False))

backed up -> /content/drive/MyDrive/7006SCN_GSOD/models/LogisticRegression
backed up -> /content/drive/MyDrive/7006SCN_GSOD/parquet/predictions/LogisticRegression
saved model + test predictions for LogisticRegression
                  feature  coefficient
         EXTREME_COUNT_3D     0.795729
               IS_EXTREME     0.642939
                     DEWP     0.353822
                  THUNDER    -0.221257
CLIMATE_ZONE_OHE_Tropical     0.195847
        SEASON_OHE_Summer     0.189509
          DEWP_DEPRESSION     0.176842
             TEMP_MEAN_3D    -0.171658
                 LATITUDE    -0.152466
                      MAX    -0.140547
                     SNDP     0.140487
                     SNOW     0.139261


In [8]:
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", weightCol="w", maxBins=64, seed=SEED)
dt_base = run_baseline("DecisionTree", dt)

[DecisionTree] BASELINE trained in 11.0s -> PR-AUC=0.6506 ROC-AUC=0.8728 recall=0.790 precision=0.521


In [9]:
dt_grid = (ParamGridBuilder()
           .addGrid(dt.maxDepth, [6, 10, 14])
           .addGrid(dt.minInstancesPerNode, [1, 100])
           .build())
dt_best = run_tuning("DecisionTree", dt, dt_grid, ["maxDepth", "minInstancesPerNode", "maxBins"])

[DecisionTree] CrossValidator: 6 configs x 3 folds in 171.9s
[DecisionTree] best = maxDepth=10, minInstancesPerNode=1, maxBins=64 | trained in 11.0s
[DecisionTree] TEST -> PR-AUC=0.7059 ROC-AUC=0.8865 F1=0.616 recall=0.797 precision=0.502


In [10]:
save_model("DecisionTree", dt_best)
print(dt_best.toDebugString[:1500])

backed up -> /content/drive/MyDrive/7006SCN_GSOD/models/DecisionTree
backed up -> /content/drive/MyDrive/7006SCN_GSOD/parquet/predictions/DecisionTree
saved model + test predictions for DecisionTree
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_32fc96cf1129, depth=10, numNodes=1289, numClasses=2, numFeatures=44
  If (feature 20 <= 0.06409956658175564)
   If (feature 10 <= 0.9430924731712508)
    If (feature 3 <= -1.7851857694052335)
     If (feature 3 <= -2.0203365863667617)
      If (feature 11 <= -2.1072709519100172)
       If (feature 16 <= 0.6689840690836395)
        If (feature 22 <= -0.6457172429176927)
         If (feature 4 <= -2.586004219164468)
          Predict: 1.0
         Else (feature 4 > -2.586004219164468)
          Predict: 0.0
        Else (feature 22 > -0.6457172429176927)
         If (feature 0 <= 0.6244690526308634)
          If (feature 7 <= -0.9053089718782974)
           If (feature 19 <= -0.3151162003139808)
            Predict: 0.0
           El

In [11]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="w", maxBins=64,
                            featureSubsetStrategy="sqrt", subsamplingRate=0.8, seed=SEED)
rf_base = run_baseline("RandomForest", rf)

[RandomForest] BASELINE trained in 17.9s -> PR-AUC=0.7203 ROC-AUC=0.8950 recall=0.786 precision=0.520


In [12]:
rf_grid = (ParamGridBuilder()
           .addGrid(rf.numTrees, [40, 80])
           .addGrid(rf.maxDepth, [8, 12])
           .build())
rf_best = run_tuning("RandomForest", rf, rf_grid, ["numTrees", "maxDepth", "featureSubsetStrategy"])

[RandomForest] CrossValidator: 4 configs x 3 folds in 1,763.9s
[RandomForest] best = numTrees=80, maxDepth=12, featureSubsetStrategy=sqrt | trained in 319.5s
[RandomForest] TEST -> PR-AUC=0.7616 ROC-AUC=0.9122 F1=0.657 recall=0.770 precision=0.573


In [13]:
save_model("RandomForest", rf_best)

backed up -> /content/drive/MyDrive/7006SCN_GSOD/models/RandomForest
backed up -> /content/drive/MyDrive/7006SCN_GSOD/parquet/predictions/RandomForest
saved model + test predictions for RandomForest


In [14]:
gbt = GBTClassifier(featuresCol="features", labelCol="label", weightCol="w", maxBins=64,
                    stepSize=0.1, subsamplingRate=0.8, seed=SEED)
gbt_base = run_baseline("GBT", gbt)

[GBT] BASELINE trained in 49.2s -> PR-AUC=0.7469 ROC-AUC=0.9085 recall=0.797 precision=0.520


In [15]:
gbt_grid = (ParamGridBuilder()
            .addGrid(gbt.maxIter, [30, 60])
            .addGrid(gbt.maxDepth, [4, 6])
            .build())
gbt_best = run_tuning("GBT", gbt, gbt_grid, ["maxIter", "maxDepth", "stepSize"])

[GBT] CrossValidator: 4 configs x 3 folds in 1,140.3s
[GBT] best = maxIter=60, maxDepth=6, stepSize=0.1 | trained in 154.1s
[GBT] TEST -> PR-AUC=0.7609 ROC-AUC=0.9148 F1=0.629 recall=0.811 precision=0.514


In [16]:
save_model("GBT", gbt_best)

backed up -> /content/drive/MyDrive/7006SCN_GSOD/models/GBT
backed up -> /content/drive/MyDrive/7006SCN_GSOD/parquet/predictions/GBT
saved model + test predictions for GBT


In [17]:
res = pd.DataFrame(RESULTS)
summary = res[res.stage.isin(["tuned", "reference"])][
    ["model", "hyperparameters", "train_time_s", "cv_time_s", "accuracy", "precision", "recall", "f1", "auc_roc", "auc_pr"]]
pd.set_option("display.width", 220)
print(summary.round(4).to_string(index=False))
save_csv(res.round(5), "task3_all_results.csv")
save_csv(summary.round(5), "task3_model_summary.csv")
save_csv(pd.DataFrame(TUNING_LOG), "task3_tuning_log.csv")

bt = res[res.stage.isin(["baseline", "tuned"])].pivot(index="model", columns="stage", values="auc_pr")
bt["gain"] = bt["tuned"] - bt["baseline"]
print("\nPR-AUC baseline -> tuned\n", bt.round(4))

              model                                      hyperparameters  train_time_s  cv_time_s  accuracy  precision  recall     f1  auc_roc  auc_pr
Persistence (naive)                                                  NaN           0.0        NaN    0.8899     0.6467  0.6466 0.6467      NaN     NaN
 LogisticRegression      regParam=0.001, elasticNetParam=0.0, maxIter=50          16.3      255.8    0.8658     0.5524  0.7330 0.6300   0.8772  0.6837
       DecisionTree       maxDepth=10, minInstancesPerNode=1, maxBins=64          11.0      171.9    0.8453     0.5024  0.7965 0.6162   0.8865  0.7059
       RandomForest numTrees=80, maxDepth=12, featureSubsetStrategy=sqrt         319.5     1763.9    0.8748     0.5733  0.7699 0.6572   0.9122  0.7616
                GBT                 maxIter=60, maxDepth=6, stepSize=0.1         154.1     1140.3    0.8508     0.5136  0.8108 0.6289   0.9148  0.7609
saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/task3_all_results.csv
saved -> /content/d

In [18]:
imp = []
for name, m in [("DecisionTree", dt_best), ("RandomForest", rf_best), ("GBT", gbt_best)]:
    for f, v in zip(feature_names, m.featureImportances.toArray()):
        imp.append({"model": name, "feature": f, "importance": float(v), "method": "Spark impurity importance"})
for f, v in zip(feature_names, lr_best.coefficients.toArray()):
    imp.append({"model": "LogisticRegression", "feature": f, "importance": float(abs(v)), "method": "|standardised coefficient|"})
imp = pd.DataFrame(imp)
save_csv(imp, "task3_feature_importance.csv")
print(imp[imp.model == "RandomForest"].sort_values("importance", ascending=False).head(10).to_string(index=False))

saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/task3_feature_importance.csv
       model          feature  importance                    method
RandomForest EXTREME_COUNT_3D    0.209484 Spark impurity importance
RandomForest       IS_EXTREME    0.197163 Spark impurity importance
RandomForest     TEMP_MEAN_3D    0.107721 Spark impurity importance
RandomForest              MAX    0.106639 Spark impurity importance
RandomForest             TEMP    0.087800 Spark impurity importance
RandomForest              MIN    0.061545 Spark impurity importance
RandomForest             DEWP    0.043245 Spark impurity importance
RandomForest  DEWP_DEPRESSION    0.016624 Spark impurity importance
RandomForest              SLP    0.013681 Spark impurity importance
RandomForest            MXSPD    0.011618 Spark impurity importance
